In [1]:
# ========== ANÁLISIS COMPLETO REPORTE SURTIDO ==========
# Script consolidado para procesar reporte de surtido y generar análisis en Excel

import os
import pandas as pd
import glob
import warnings
from datetime import datetime
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment

# Suprimir advertencias
warnings.filterwarnings('ignore')

# ========== CONFIGURACIÓN ==========
carpeta_archivos = r"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Limpieza TiemposRuta Abastos\Reporte Surtido"
ruta_salida = r"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Limpieza TiemposRuta Abastos\Outputs"

# ========== DICCIONARIO DE PRODUCTOS ==========
productos_dict = {
    40081: {"Desc": "PVC-101", "Pza": 935, "Tipo": "Móvil"},
    40082: {"Desc": "PVC-102", "Pza": 560, "Tipo": "Móvil"},
    40083: {"Desc": "PVC-103", "Pza": 385, "Tipo": "Móvil"},
    40084: {"Desc": "PVC-104", "Pza": 270, "Tipo": "Móvil"},
    40085: {"Desc": "PVC-105", "Pza": 190, "Tipo": "Móvil"},
    40086: {"Desc": "PVC-106", "Pza": 120, "Tipo": "Móvil"},
    43038: {"Desc": "PVC-001", "Pza": 880, "Tipo": "Móvil"},
    43039: {"Desc": "PVC-002", "Pza": 540, "Tipo": "Móvil"},
    43040: {"Desc": "PVC-003", "Pza": 385, "Tipo": "Móvil"},
    45042: {"Desc": "CPVC-001", "Pza": 1600, "Tipo": "Móvil"},
    45043: {"Desc": "CPVC-002", "Pza": 880, "Tipo": "Móvil"},
    45044: {"Desc": "CPVC-003", "Pza": 520, "Tipo": "Móvil"},
    45520: {"Desc": "PVC-004", "Pza": 240, "Tipo": "Móvil"},
    45521: {"Desc": "PVC-005", "Pza": 190, "Tipo": "Móvil"},
    45522: {"Desc": "PVC-006", "Pza": 120, "Tipo": "Móvil"},
    45560: {"Desc": "CPVC-004", "Pza": 385, "Tipo": "Móvil"},
    45561: {"Desc": "CPVC-005", "Pza": 270, "Tipo": "Móvil"},
    45562: {"Desc": "CPVC-006", "Pza": 140, "Tipo": "Móvil"},
    49897: {"Desc": "CV-001", "Pza": 960, "Tipo": "4m"},
    49898: {"Desc": "CV-002", "Pza": 600, "Tipo": "4m"},
    49899: {"Desc": "CV-003", "Pza": 345, "Tipo": "4m"},
    45444: {"Desc": "CV-004", "Pza": 225, "Tipo": "4m"},
    45445: {"Desc": "CV-005", "Pza": 140, "Tipo": "4m"},
    45446: {"Desc": "CV-006", "Pza": 90, "Tipo": "4m"}
}

# Crear DataFrame de productos
productos_df = pd.DataFrame.from_dict(productos_dict, orient='index')
productos_df.index.name = 'CLAVEPRODUCTO'
productos_df = productos_df.reset_index()

# ========== LEER TODOS LOS ARCHIVOS DE SURTIDO ==========
patrones_archivos = [
    os.path.join(carpeta_archivos, "*.xlsx"),
    os.path.join(carpeta_archivos, "*.xls"), 
    os.path.join(carpeta_archivos, "*.csv")
]

archivos_encontrados = []
for patron in patrones_archivos:
    archivos_encontrados.extend(glob.glob(patron))

archivos_encontrados = sorted(archivos_encontrados)

if len(archivos_encontrados) == 0:
    raise FileNotFoundError("No se encontraron archivos en la carpeta especificada")

df_surtido_lista = []
errores_lectura = []

for archivo in archivos_encontrados:
    try:
        if archivo.lower().endswith('.csv'):
            try:
                df_temp = pd.read_csv(archivo, encoding='utf-8-sig')
            except:
                try:
                    df_temp = pd.read_csv(archivo, encoding='latin1')
                except:
                    df_temp = pd.read_csv(archivo, encoding='cp1252')
        else:
            df_temp = pd.read_excel(archivo)

        df_surtido_lista.append(df_temp)
    except Exception as e:
        errores_lectura.append(f"{os.path.basename(archivo)}: {e}")

if len(df_surtido_lista) == 0:
    detalle = "\n".join(errores_lectura) if len(errores_lectura) > 0 else "Sin detalle"
    raise ValueError(f"No se pudo leer ningún archivo válido.\n{detalle}")

df_surtido = pd.concat(df_surtido_lista, ignore_index=True)
print(f"Archivos encontrados: {len(archivos_encontrados)} | Archivos procesados: {len(df_surtido_lista)}")
if len(errores_lectura) > 0:
    print("Archivos con error de lectura:")
    for err in errores_lectura:
        print(f"- {err}")

# ========== REALIZAR MERGE ==========
# Filtrar solo filas con CLAVEPRODUCTO numérico
filtro_numerico = df_surtido['CLAVEPRODUCTO'].apply(lambda x: str(x).strip().isdigit())
df_surtido = df_surtido[filtro_numerico].copy()
df_surtido['CLAVEPRODUCTO'] = df_surtido['CLAVEPRODUCTO'].astype(int)
productos_df['CLAVEPRODUCTO'] = productos_df['CLAVEPRODUCTO'].astype(int)

df_analisis = df_surtido.merge(
    productos_df[['CLAVEPRODUCTO', 'Desc', 'Pza', 'Tipo']], 
    on='CLAVEPRODUCTO', 
    how='left'
)

# Crear DataFrame final con columnas específicas
columnas_finales = ['RUTA', 'CLAVEPRODUCTO', 'CANTIDADSURTIDA', 'Tipo', 'Desc', 'Pza']
df_final = df_analisis[columnas_finales].copy()

# ========== CREAR ARCHIVO EXCEL ==========
# Verificar y crear carpeta de salida
if not os.path.exists(ruta_salida):
    os.makedirs(ruta_salida, exist_ok=True)

# Crear nombre del archivo con timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
nombre_archivo = f"Analisis_Surtido_{timestamp}.xlsx"
ruta_completa = os.path.join(ruta_salida, nombre_archivo)

# ========== PROCESAR DATAFRAME FINAL ==========
# Trabajar directamente con df_final sin agrupar

# ========== CREAR COLUMNA CUENTA DE CANTIDAD SURTIDA ==========
# Contar cuántas veces se surtió cada cantidad específica por RUTA y CLAVEPRODUCTO
cuenta_cantidad = df_final.groupby(['RUTA', 'CLAVEPRODUCTO', 'CANTIDADSURTIDA']).size().reset_index(name='Cuenta_Cantidad_Surtida')

# Hacer merge para agregar la cuenta al DataFrame final
df_agrupado = df_final.merge(
    cuenta_cantidad,
    on=['RUTA', 'CLAVEPRODUCTO', 'CANTIDADSURTIDA'],
    how='left'
)

# Crear columna Ruta Conca (año + ruta) en formato numérico
año_actual = datetime.now().year
df_agrupado['Ruta Conca'] = (año_actual * 100000 + df_agrupado['RUTA']).astype(int)

# Reordenar columnas para que Ruta Conca vaya antes de RUTA y incluir la nueva columna
columnas_ordenadas = ['Ruta Conca', 'RUTA', 'CLAVEPRODUCTO', 'CANTIDADSURTIDA', 'Cuenta_Cantidad_Surtida', 'Tipo', 'Desc', 'Pza']
df_agrupado = df_agrupado[columnas_ordenadas]

# ========== FILTRAR FILAS DONDE CANTIDADSURTIDA Y Pza COINCIDAN ==========
# Mantener solo las filas donde CANTIDADSURTIDA es igual a Pza
df_agrupado = df_agrupado[df_agrupado['CANTIDADSURTIDA'] == df_agrupado['Pza']]

# ========== AGRUPAR FILAS DUPLICADAS ==========
# Agrupar por todas las columnas para eliminar duplicados completos
columnas_agrupacion = ['Ruta Conca', 'RUTA', 'CLAVEPRODUCTO', 'CANTIDADSURTIDA', 'Cuenta_Cantidad_Surtida', 'Tipo', 'Desc', 'Pza']
df_agrupado = df_agrupado.groupby(columnas_agrupacion).size().reset_index(name='Registros_Duplicados')

# Seleccionar y reordenar las columnas en el orden deseado
columnas_finales = ['Ruta Conca', 'RUTA', 'Cuenta_Cantidad_Surtida', 'CLAVEPRODUCTO', 'Tipo', 'CANTIDADSURTIDA']
df_agrupado = df_agrupado[columnas_finales]

# ========== CREAR ARCHIVO EXCEL CON SOLO EL DATAFRAME FINAL ==========
# Crear Excel con una sola hoja con el DataFrame agrupado final
with pd.ExcelWriter(ruta_completa, engine='openpyxl') as writer:
    df_agrupado.to_excel(writer, sheet_name='Analisis_Final', index=False)

# ========== APLICAR FORMATO BÁSICO ==========
wb = openpyxl.load_workbook(ruta_completa)
header_font = Font(bold=True, color="FFFFFF")
header_fill = PatternFill(start_color="4472C4", end_color="4472C4", fill_type="solid")

ws = wb['Analisis_Final']

# Formatear solo los encabezados (primera fila)
for cell in ws[1]:
    if cell.value:
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = Alignment(horizontal="center")

# Ajustar ancho de columnas
for column in ws.columns:
    max_length = 0
    column_letter = column[0].column_letter
    for cell in column:
        try:
            if len(str(cell.value)) > max_length:
                max_length = len(str(cell.value))
        except:
            pass
    adjusted_width = min(max_length + 2, 50)
    ws.column_dimensions[column_letter].width = adjusted_width

wb.save(ruta_completa)

# ========== MOSTRAR RESULTADOS ==========
display(df_agrupado.head(20))
print("✅ Archivo listo")

Archivos encontrados: 1 | Archivos procesados: 1


,Ruta Conca,RUTA,Cuenta_Cantidad_Surtida,CLAVEPRODUCTO,Tipo,CANTIDADSURTIDA
0,202672377,72377,2,45042,Móvil,1600
1,202672388,72388,3,43040,Móvil,385
2,202672401,72401,1,49897,4m,960
3,202672401,72401,1,49898,4m,600
4,202672402,72402,1,43038,Móvil,880
5,202672421,72421,3,45043,Móvil,880
6,202672428,72428,1,45044,Móvil,520
7,202672428,72428,1,45521,Móvil,190
8,202672428,72428,3,49898,4m,600
9,202672430,72430,1,43039,Móvil,540


✅ Archivo listo
